In [ ]:
import requests
import pandas as pd
import re

In [ ]:
API_URL = ##############
headers = ##############

In [ ]:
def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    response
    return response.json()


In [ ]:
def remove_newlines(text):
    return re.sub(r'[\n\r]+', '', text)

In [ ]:

sentences = pd.read_csv('books_abs.csv', sep='^', header=None)

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
final_df = pd.DataFrame(columns=['text'])
for i, row in enumerate(sentences.iterrows()):
    sentences_ls = []
    for sentence in row[1]:
        sentence = str(sentence)
        if sentence != 'nan':
            prompt = {
                "inputs": (f"Simplify the following so that a child will understand: {sentence}. Focus on this task only and the structure of your answer should be sentence format. Do not use new-line characters, colons, semi-colons or bullet points."),
                "parameters": {
                    "max_input_tokens": 1024,
                    "max_output_tokens": 256,
                    "temperature": 0.7,
                }
            }
            output = query(prompt)
            print('output: ',output)
            generated_text = output[0].get('generated_text', '').strip()
            simplified_sentence = generated_text.split('points.', 1)
            if len(simplified_sentence) > 1:
                simplified_sentence = simplified_sentence[1].strip()
            
            simplified_sentence = remove_newlines(simplified_sentence)

            sentences_ls.append(simplified_sentence)
        
    combined_text = ' '.join(sentences_ls)
    final_text = pd.DataFrame({'text': [combined_text]})
    final_df = pd.concat([final_df, final_text], axis=0)

final_df.to_csv(f'llama_books_abstracts.csv', index=False, header=True, sep='^')